# Breast Cancer Classification using Machine Learning
     by Divyansh Srivastav

## Project Introduction

In this project, I built a machine learning pipeline to classify breast cancer tumors as **malignant** or **benign** using the Breast Cancer dataset available in scikit-learn.

The goal of this project is to explore how **feature engineering and different feature selection techniques** can impact the performance of machine learning models.

During this project, I implemented a complete machine learning workflow including:

- Feature engineering to create additional meaningful variables
- Data preprocessing using pipelines
- Feature selection using multiple techniques
- Model training using Random Forest
- Model evaluation using Stratified K-Fold cross validation

This project demonstrates how structured preprocessing pipelines and feature selection methods can improve model performance and create a more reliable predictive model.

## Tools and Libraries
- Python
- NumPy
- Pandas
- Scikit-learn

## Objective
To compare different feature selection techniques and evaluate their impact on classification accuracy.


In [8]:
|import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, KBinsDiscretizer, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif, RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

## Feature Engineering

In this step, I create additional features from the original dataset to help the machine learning model capture more meaningful patterns.

Feature engineering is an important step in many machine learning projects because well-designed features can significantly improve model performance.

For this project, I implemented several feature engineering techniques such as:
- Creating interaction features between important variables
- Generating polynomial transformations
- Applying log transformations to handle skewed distributions
- Converting continuous values into categorical bins
- Creating ratio-based features to capture relationships between variables

These engineered features are added to the dataset before applying preprocessing and training the machine learning models.


In [9]:
def create_engineered_features(df):
    """Add a few engineered features to illustrate feature creation."""
    # 1. Interaction term: mean radius * mean texture
    df['radius_texture_interaction'] = df['mean radius'] * df['mean texture']

    # 2. Polynomial feature (square) of mean perimeter
    df['perimeter_sq'] = df['mean perimeter'] ** 2

    # 3. Log transform (safe) of mean area
    df['log_area'] = np.log1p(df['mean area'])

    # 4. Binned categorical from mean smoothness (3 bins)
    kb = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
    df['smoothness_bin'] = kb.fit_transform(df[['mean smoothness']]).astype(int)

    # 5. Ratio feature (concavity / concave points) with safe-guard
    eps = 1e-6
    df['concavity_over_concave_points'] = df['mean concave points'] / (df['mean concavity'] + eps)

    return df

## Data Preprocessing

Before training machine learning models, the dataset needs to be prepared so that the algorithms can learn effectively from the data.

In this step, I created a preprocessing pipeline to handle common data preparation tasks in a structured and reproducible way.

The preprocessing pipeline performs the following operations:

- Handling missing values using median imputation
- Standardizing numerical features using feature scaling

Using preprocessing pipelines helps maintain a clean workflow and ensures that the same transformations are consistently applied during both training and evaluation of the model.


In [10]:
def build_preprocessor(numeric_features):
    numeric_transformer = Pipeline(steps=[
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_features)
    ], remainder='drop')

    return preprocessor

## Model Evaluation

Once the machine learning pipeline is built, it is important to measure how well the models perform.

In this project, I use **Stratified K-Fold Cross Validation** to evaluate the models. This technique:

- Splits the dataset into multiple folds while preserving the class distribution
- Trains the model on different folds and validates it on the remaining fold
- Provides a reliable estimate of model performance

Using cross-validation helps ensure that the results are robust and not dependent on a single random train-test split. The performance is measured using **accuracy**, along with the standard deviation across folds to understand variability.


In [11]:
def evaluate_pipeline(pipeline, X, y, cv):
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    return scores.mean(), scores.std()

## Model Training and Feature Selection

With the dataset prepared and features engineered, I train machine learning models to classify breast cancer tumors as **malignant** or **benign**.

In this project, I also explore how different **feature selection techniques** affect model performance. Feature selection helps:

- Reduce the number of input variables
- Improve model interpretability
- Remove irrelevant or noisy features
- Potentially improve accuracy

The methods used in this project include:

1. **Baseline (all features)** – Train a model using all available features to serve as a reference point.  
2. **Filter Method: SelectKBest** – Select features based on univariate statistical tests.  
3. **Wrapper Method: Recursive Feature Elimination (RFE)** – Iteratively removes the least important features based on model performance.  
4. **Embedded Method: SelectFromModel** – Select features based on importance weights from a Random Forest classifier.  

The model used throughout is a **Random Forest Classifier**, and performance is evaluated using **Stratified K-Fold Cross Validation**.  
This approach allows me to compare different feature selection strategies and identify which features contribute most to the model’s predictive power.


In [7]:
def main():
    # Load dataset
    data = load_breast_cancer(as_frame=True)
    X = data.frame.drop(columns=['target'])
    y = data.target

    # Start from original features
    df = X.copy()

    # Create engineered features
    df = create_engineered_features(df)

    # Feature list
    all_features = list(df.columns)

    # Preprocessor
    preprocessor = build_preprocessor(all_features)

    # Classifier used for evaluation (stable baseline)
    clf = RandomForestClassifier(n_estimators=200, random_state=42)

    # Baseline pipeline (all features)
    pipeline_baseline = Pipeline(steps=[('pre', preprocessor), ('clf', clf)])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print('Evaluating baseline (all features)...')
    baseline_mean, baseline_std = evaluate_pipeline(pipeline_baseline, df, y, cv)
    print(f'Baseline accuracy: {baseline_mean:.4f} +/- {baseline_std:.4f}\n')

    # 1) Filter method: SelectKBest (ANOVA f-test)
    k = 12
    skb = SelectKBest(score_func=f_classif, k=k)
    pipeline_skb = Pipeline(steps=[('pre', preprocessor), ('skb', skb), ('clf', clf)])
    print(f'Evaluating SelectKBest (f_classif), k={k}...')
    skb_mean, skb_std = evaluate_pipeline(pipeline_skb, df, y, cv)
    print(f'SelectKBest accuracy: {skb_mean:.4f} +/- {skb_std:.4f}\n')
    preprocessed = preprocessor.fit_transform(df)
    skb.fit(preprocessed, y)
    # Note: ColumnTransformer drops column names; we'll compute scores using preprocessor on dataframe columns
    # Simpler: compute scores using raw df columns (since our preprocessor only scales)
    skb_feature_indices = skb.get_support(indices=True)
    selected_by_skb = [all_features[i] for i in skb_feature_indices]
    print('SelectKBest selected features:', selected_by_skb, '\n')

    # 2) Wrapper method: RFE with LogisticRegression
    base_lr = LogisticRegression(solver='liblinear', max_iter=200)
    rfe = RFE(estimator=base_lr, n_features_to_select=10, step=1)
    pipeline_rfe = Pipeline(steps=[('pre', preprocessor), ('rfe', rfe), ('clf', clf)])
    print('Evaluating RFE (LogisticRegression) selecting 10 features...')
    rfe_mean, rfe_std = evaluate_pipeline(pipeline_rfe, df, y, cv)
    print(f'RFE accuracy: {rfe_mean:.4f} +/- {rfe_std:.4f}\n')

    # Fit RFE on full data to get selected features
    # Because RFE expects numeric array, transform first
    preproc_X = preprocessor.fit_transform(df)
    rfe.fit(preproc_X, y)
    rfe_support = rfe.get_support()
    selected_by_rfe = [all_features[i] for i, keep in enumerate(rfe_support) if keep]
    print('RFE selected features:', selected_by_rfe, '\n')

    # 3) Embedded / model-based selection: RandomForest importance -> SelectFromModel
    selector = SelectFromModel(estimator=RandomForestClassifier(n_estimators=300, random_state=0), threshold='median')
    pipeline_sfm = Pipeline(steps=[('pre', preprocessor), ('sfm', selector), ('clf', clf)])
    print('Evaluating SelectFromModel (RandomForest importance, threshold=median) ...')
    sfm_mean, sfm_std = evaluate_pipeline(pipeline_sfm, df, y, cv)
    print(f'SelectFromModel accuracy: {sfm_mean:.4f} +/- {sfm_std:.4f}\n')

    # Fit selector to get selected features names
    selector.fit(preproc_X, y)
    sfm_support = selector.get_support()
    selected_by_sfm = [all_features[i] for i, keep in enumerate(sfm_support) if keep]
    print('SelectFromModel selected features:', selected_by_sfm, '\n')

    results = pd.DataFrame({
        'method': ['baseline_all', 'select_kbest', 'rfe', 'select_from_model'],
        'mean_accuracy': [baseline_mean, skb_mean, rfe_mean, sfm_mean],
        'std_accuracy': [baseline_std, skb_std, rfe_std, sfm_std]
    })

    print('\nSummary results:')
    print(results.sort_values('mean_accuracy', ascending=False).to_string(index=False))
    print('\nNotes:')
    print('- Try recursive selection + hyperparameter tuning of the estimator for best results.')
    print('- Try different feature creation ideas (domain-specific) and interaction terms.')
    print('- For high-dimensional data, consider L1-based selection or dimensionality reduction (PCA).')


if __name__ == '__main__':
    main()

Evaluating baseline (all features)...
Baseline accuracy: 0.9649 +/- 0.0055

Evaluating SelectKBest (f_classif), k=12...
SelectKBest accuracy: 0.9543 +/- 0.0151

SelectKBest selected features: ['mean radius', 'mean perimeter', 'mean area', 'mean concavity', 'mean concave points', 'worst radius', 'worst perimeter', 'worst area', 'worst concave points', 'radius_texture_interaction', 'perimeter_sq', 'log_area'] 

Evaluating RFE (LogisticRegression) selecting 10 features...
RFE accuracy: 0.9508 +/- 0.0180

RFE selected features: ['mean concave points', 'radius error', 'area error', 'compactness error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst concavity', 'worst concave points'] 

Evaluating SelectFromModel (RandomForest importance, threshold=median) ...
SelectFromModel accuracy: 0.9561 +/- 0.0200

SelectFromModel selected features: ['mean radius', 'mean perimeter', 'mean area', 'mean compactness', 'mean concavity', 'mean concave points', 'radius error', 'area

## Project Results and Insights

After evaluating multiple feature selection techniques with a Random Forest classifier, the following results were obtained:

### Baseline (All Features)
- Accuracy: **96.5% ± 0.55%**
- Using all features provides a strong reference point for comparison.

### SelectKBest (Filter Method)
- Accuracy: **95.4% ± 1.51%**
- Selected features:
  `['mean radius', 'mean perimeter', 'mean area', 'mean concavity', 'mean concave points', 'worst radius', 'worst perimeter', 'worst area', 'worst concave points', 'radius_texture_interaction', 'perimeter_sq', 'log_area']`

### RFE (Wrapper Method with Logistic Regression)
- Accuracy: **95.1% ± 1.80%**
- Selected features:
  `['mean concave points', 'radius error', 'area error', 'compactness error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst concavity', 'worst concave points']`

### SelectFromModel (Embedded Method using Random Forest)
- Accuracy: **95.6% ± 2.0%**
- Selected features:
  `['mean radius', 'mean perimeter', 'mean area', 'mean compactness', 'mean concavity', 'mean concave points', 'radius error', 'area error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst concavity', 'worst concave points', 'radius_texture_interaction', 'perimeter_sq', 'log_area']`

### Insights
- Using all features (baseline) achieved the **highest accuracy**, but feature selection methods provide a smaller set of important features for interpretability.  
- SelectFromModel captured **the most important features based on model importance**, balancing accuracy and feature reduction.  
- RFE and SelectKBest also reduced features but with slightly lower accuracy.  
- Feature engineering clearly contributed to improving predictive performance.  

### Recommendations
- Explore **hyperparameter tuning** of classifiers to improve performance further.  
- Experiment with **additional feature creation**, including domain-specific interactions.  
- For higher-dimensional datasets, consider **L1-based feature selection** or **dimensionality reduction (PCA)**.
